# 🚂 Sysslan IT Solutions — Internship Project
## Level 3: Data Exploration (EDA)
**Project:** Train Route Analysis and Journey Time Prediction  
**Level:** 3 — Exploratory Data Analysis  
**Dataset:** Indian Railways Train Schedule  
---

## 📦 Step 0: Import Libraries & Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')

print('✅ Libraries imported!')

✅ Libraries imported!


In [2]:
# Load cleaned master data from Level 2
train_master = pd.read_csv('train_master_cleaned.csv')
df_clean     = pd.read_csv('dataset_cleaned.csv')

print(f'✅ train_master loaded : {train_master.shape[0]:,} trains × {train_master.shape[1]} features')
print(f'✅ df_clean loaded     : {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns')
print()
train_master.head()

✅ train_master loaded : 10,625 trains × 12 features
✅ df_clean loaded     : 186,074 rows × 16 columns



,Train_No,Journey_Duration_Min,Journey_Duration_Hrs,Total_Distance_km,Num_Stops,Source_Station,Source_Code,Destination_Station,Destination_Code,Departure_Time_Source,Arrival_Time_Dest,Avg_Speed_kmh
0,107,105.0,1.75,78,4,SAWANTWADI R,SWV,MADGOAN JN.,MAO,10:25,12:10,44.57
1,108,115.0,1.92,83,4,MADGOAN JN.,MAO,SAWANTWADI R,SWV,20:30,22:25,43.23
2,128,1325.0,22.08,978,22,MADGOAN JN.,MAO,CHHATRAPATI,KOP,19:40,17:45,44.29
3,401,750.0,12.50,1618,12,AURANGABAD,AWB,VARANASI JN.,BSB,21:30,10:00,129.44
4,421,540.0,9.00,1276,5,LUCKNOW JN.,LKO,SHRI MATA VA,SVDK,23:00,08:00,141.78


---
## 📊 Task 3.1 — Compare Journey Duration Across Different Routes

In [3]:
# Create a Route label: Source → Destination
train_master['Route'] = train_master['Source_Station'].str.strip() + ' → ' + train_master['Destination_Station'].str.strip()

# Categorize trains by journey duration
def duration_category(hrs):
    if hrs <= 2:   return '🟢 Short (≤2 hrs)'
    elif hrs <= 8: return '🟡 Medium (2–8 hrs)'
    elif hrs <= 18:return '🟠 Long (8–18 hrs)'
    else:          return '🔴 Very Long (>18 hrs)'

train_master['Duration_Category'] = train_master['Journey_Duration_Hrs'].apply(duration_category)

# Count per category
cat_counts = train_master['Duration_Category'].value_counts().reset_index()
cat_counts.columns = ['Category', 'Count']

print('📊 Journey Duration Category Distribution:')
print('=' * 45)
for _, row in cat_counts.iterrows():
    pct = row['Count'] / len(train_master) * 100
    print(f"  {row['Category']:<25} : {row['Count']:>5,} trains ({pct:.1f}%)")

📊 Journey Duration Category Distribution:
  🟢 Short (≤2 hrs)          : 5,184 trains (48.8%)
  🟡 Medium (2–8 hrs)        : 3,339 trains (31.4%)
  🟠 Long (8–18 hrs)         : 1,661 trains (15.6%)
  🔴 Very Long (>18 hrs)     :   441 trains (4.2%)


In [4]:
# Top 10 Longest Routes by Journey Duration
print('🔴 Top 10 Longest Journey Trains:')
print('=' * 75)
longest = train_master.nlargest(10, 'Journey_Duration_Hrs')[
    ['Train_No', 'Source_Station', 'Destination_Station', 
     'Journey_Duration_Hrs', 'Total_Distance_km', 'Num_Stops']
].reset_index(drop=True)
longest.index += 1
print(longest.to_string())

🔴 Top 10 Longest Journey Trains:
    Train_No Source_Station Destination_Station  Journey_Duration_Hrs  Total_Distance_km  Num_Stops
1      11103      JHANSI JN        BANDRA TERMI                 23.92               1321         21
2      12949      PORBANDAR         SANTRAGACHI                 23.92               2664         37
3      15001    MUZAFFARPUR           DEHRA DUN                 23.92               1122         26
4       8617         RANCHI         ANAND VIHAR                 23.83               1365         15
5      12494   HAZRAT NIZAM            PUNE JN.                 23.83               1501         10
6      12766       AMRAVATI            TIRUPATI                 23.83               1305         24
7      12778      KOCHUVELI           HUBLI JN.                 23.83               1297         22
8      11404    CHHATRAPATI        NAGPUR JN.(C                 23.75               1026         22
9      12777      HUBLI JN.           KOCHUVELI                 23.

In [5]:
# Top 10 Shortest Routes
print('🟢 Top 10 Shortest Journey Trains:')
print('=' * 75)
shortest = train_master.nsmallest(10, 'Journey_Duration_Hrs')[
    ['Train_No', 'Source_Station', 'Destination_Station',
     'Journey_Duration_Hrs', 'Total_Distance_km', 'Num_Stops']
].reset_index(drop=True)
shortest.index += 1
print(shortest.to_string())

🟢 Top 10 Shortest Journey Trains:
    Train_No Source_Station Destination_Station  Journey_Duration_Hrs  Total_Distance_km  Num_Stops
1      96001         KARJAT           PALASDARI                  0.08                  2          2
2      96005         KARJAT           PALASDARI                  0.08                  2          2
3      96007         KARJAT           PALASDARI                  0.08                  2          2
4      96009         KARJAT           PALASDARI                  0.08                  2          2
5      96011         KARJAT           PALASDARI                  0.08                  2          2
6      96013         KARJAT           PALASDARI                  0.08                  2          2
7      96015         KARJAT           PALASDARI                  0.08                  2          2
8      96017         KARJAT           PALASDARI                  0.08                  2          2
9      96002      PALASDARI              KARJAT                  0

In [6]:
# Duration stats by category
print('📋 Duration Statistics by Category:')
train_master.groupby('Duration_Category')['Journey_Duration_Hrs'].describe().round(2)

📋 Duration Statistics by Category:


,count,mean,std,min,25%,50%,75%,max
Duration_Category,,,,,,,,
🔴 Very Long (>18 hrs),441.0,21.07,1.68,18.08,19.67,21.17,22.50,23.92
🟠 Long (8–18 hrs),1661.0,12.52,2.86,8.08,9.92,12.25,15.00,18.00
🟡 Medium (2–8 hrs),3339.0,4.11,1.71,2.02,2.58,3.58,5.42,8.00
🟢 Short (≤2 hrs),5184.0,1.06,0.48,0.08,0.75,1.00,1.42,2.00


---
## 🚉 Task 3.2 — Identify Stations with High and Low Train Traffic

In [7]:
# Count how many unique trains stop at each station
station_traffic = df_clean.groupby(['Station_Code', 'Station_Name'])['Train_No'] \
    .nunique().reset_index()
station_traffic.columns = ['Station_Code', 'Station_Name', 'Train_Count']
station_traffic = station_traffic.sort_values('Train_Count', ascending=False).reset_index(drop=True)

print(f'✅ Total unique stations analyzed: {len(station_traffic):,}')
print()

# Top 15 busiest stations
print('🔥 Top 15 Busiest Stations (High Traffic):')
print('=' * 55)
top15 = station_traffic.head(15)
for i, row in top15.iterrows():
    bar = '█' * (row['Train_Count'] // 50)
    print(f"  {i+1:>2}. {row['Station_Name']:<20} ({row['Station_Code']}) — {row['Train_Count']:>4} trains  {bar}")

✅ Total unique stations analyzed: 8,150

🔥 Top 15 Busiest Stations (High Traffic):
   1. CST-MUMBAI           (CSMT) — 1027 trains  ████████████████████
   2. KALYAN JN            (KYN) —  828 trains  ████████████████
   3. THANE                (TNA) —  796 trains  ███████████████
   4. SEALDAH              (SDAH) —  745 trains  ██████████████
   5. CHENNAI BEAC         (MSB) —  738 trains  ██████████████
   6. HOWRAH JN.           (HWH) —  699 trains  █████████████
   7. DADAR                (DR) —  567 trains  ███████████
   8. DUM DUM JN.          (DDJ) —  463 trains  █████████
   9. KURLA                (CLA) —  462 trains  █████████
  10. TAMBARAM             (TBM) —  434 trains  ████████
  11. VIJAYWADA JN         (BZA) —  416 trains  ████████
  12. GHAT KOPAR           (GC) —  406 trains  ████████
  13. BIDHANNAGAR          (BNXR) —  404 trains  ████████
  14. PANVEL               (PNVL) —  400 trains  ████████
  15. MULUND               (MLND) —  393 trains  ███████


In [8]:
# Bottom 10 least busy stations
print('🔕 Bottom 10 Least Busy Stations (Low Traffic):')
print('=' * 55)
bottom10 = station_traffic.tail(10).reset_index(drop=True)
for i, row in bottom10.iterrows():
    print(f"  {i+1:>2}. {row['Station_Name']:<25} ({row['Station_Code']}) — {row['Train_Count']} train(s)")

🔕 Bottom 10 Least Busy Stations (Low Traffic):
   1. MOTA SURKA                (MTSK) — 1 train(s)
   2. CHITODA                   (CHJC) — 1 train(s)
   3. HARISING                  (HRNS) — 1 train(s)
   4. AWAPUR F                  (AWPR) — 1 train(s)
   5. AYANDUR                   (AYD) — 1 train(s)
   6. PARANUR                   (PWU) — 1 train(s)
   7. BOLWAD                    (BLWD) — 1 train(s)
   8. SONAUILM                  (LIM) — 1 train(s)
   9. KURVA                     (KURV) — 1 train(s)
  10. BAMANI                    (BMNI) — 1 train(s)


In [9]:
# Traffic distribution summary
print('📊 Station Traffic Distribution:')
print('=' * 45)
print(f"  Stations with only 1 train  : {(station_traffic['Train_Count'] == 1).sum():>5,}")
print(f"  Stations with 2–10 trains   : {station_traffic['Train_Count'].between(2,10).sum():>5,}")
print(f"  Stations with 11–50 trains  : {station_traffic['Train_Count'].between(11,50).sum():>5,}")
print(f"  Stations with 51–200 trains : {station_traffic['Train_Count'].between(51,200).sum():>5,}")
print(f"  Stations with 200+ trains   : {(station_traffic['Train_Count'] > 200).sum():>5,}")
print(f"  Mean trains per station     : {station_traffic['Train_Count'].mean():>8.2f}")
print(f"  Max trains (busiest)        : {station_traffic['Train_Count'].max():>5,}")

📊 Station Traffic Distribution:
  Stations with only 1 train  :    56
  Stations with 2–10 trains   : 4,493
  Stations with 11–50 trains  : 2,814
  Stations with 51–200 trains :   675
  Stations with 200+ trains   :   112
  Mean trains per station     :    22.83
  Max trains (busiest)        : 1,027


---
## 📈 Task 3.3 — Relationship Between Distance and Journey Duration

In [10]:
# Pearson correlation
corr_dist_dur  = train_master['Total_Distance_km'].corr(train_master['Journey_Duration_Min'])
corr_stops_dur = train_master['Num_Stops'].corr(train_master['Journey_Duration_Min'])
corr_dist_stops = train_master['Total_Distance_km'].corr(train_master['Num_Stops'])
corr_speed_dur  = train_master['Avg_Speed_kmh'].corr(train_master['Journey_Duration_Min'])

print('📊 Pearson Correlation Analysis:')
print('=' * 55)
print(f'  Distance  vs Journey Duration : {corr_dist_dur:.4f}  ⭐ Strong positive')
print(f'  Num Stops vs Journey Duration : {corr_stops_dur:.4f}  ✅ Moderate positive')
print(f'  Distance  vs Num Stops        : {corr_dist_stops:.4f}  ✅ Moderate positive')
print(f'  Avg Speed vs Journey Duration : {corr_speed_dur:.4f}  (mixed)')
print()
print('  Interpretation:')
print('  → Distance is the STRONGEST predictor of journey duration (r=0.80)')
print('  → More stops = longer journey, but weaker than distance (r=0.54)')
print('  → Both features will be used in the ML model (Level 5)')

📊 Pearson Correlation Analysis:
  Distance  vs Journey Duration : 0.8025  ⭐ Strong positive
  Num Stops vs Journey Duration : 0.5406  ✅ Moderate positive
  Distance  vs Num Stops        : 0.4471  ✅ Moderate positive
  Avg Speed vs Journey Duration : 0.4261  (mixed)

  Interpretation:
  → Distance is the STRONGEST predictor of journey duration (r=0.80)
  → More stops = longer journey, but weaker than distance (r=0.54)
  → Both features will be used in the ML model (Level 5)


In [11]:
# Correlation matrix
print('📋 Full Correlation Matrix:')
corr_cols = ['Journey_Duration_Min', 'Journey_Duration_Hrs',
             'Total_Distance_km', 'Num_Stops', 'Avg_Speed_kmh']
train_master[corr_cols].corr().round(4)

📋 Full Correlation Matrix:


,Journey_Duration_Min,Journey_Duration_Hrs,Total_Distance_km,Num_Stops,Avg_Speed_kmh
Journey_Duration_Min,1.0000,1.0000,0.8025,0.5406,0.4261
Journey_Duration_Hrs,1.0000,1.0000,0.8025,0.5406,0.4261
Total_Distance_km,0.8025,0.8025,1.0000,0.4471,0.8241
Num_Stops,0.5406,0.5406,0.4471,1.0000,0.2969
Avg_Speed_kmh,0.4261,0.4261,0.8241,0.2969,1.0000


In [12]:
# Binned analysis: Distance buckets vs Average Duration
bins   = [0, 100, 300, 600, 1000, 2000, 5000]
labels = ['0–100km', '100–300km', '300–600km', '600–1000km', '1000–2000km', '2000km+']
train_master['Distance_Bucket'] = pd.cut(train_master['Total_Distance_km'], bins=bins, labels=labels)

bucket_analysis = train_master.groupby('Distance_Bucket', observed=True).agg(
    Train_Count=('Train_No', 'count'),
    Avg_Duration_Hrs=('Journey_Duration_Hrs', 'mean'),
    Avg_Speed_kmh=('Avg_Speed_kmh', 'mean'),
    Avg_Stops=('Num_Stops', 'mean')
).round(2)

print('📊 Distance Bucket Analysis:')
print('=' * 65)
print(bucket_analysis.to_string())

📊 Distance Bucket Analysis:
                 Train_Count  Avg_Duration_Hrs  Avg_Speed_kmh  Avg_Stops
Distance_Bucket                                                         
0–100km                 6082              1.29          34.31      10.67
100–300km               2149              4.45          38.96      20.93
300–600km                927              9.80          46.72      24.69
600–1000km               644             15.30          51.42      22.90
1000–2000km              573             15.00         125.37      28.32
2000km+                  250             17.46         141.04      33.14


---
## 💡 Task 3.4 — Key Exploratory Insights Summary

In [13]:
# Fastest trains (highest avg speed)
print('⚡ Top 10 Fastest Trains (by Avg Speed):')
print('=' * 70)
fastest = train_master.nlargest(10, 'Avg_Speed_kmh')[
    ['Train_No', 'Source_Station', 'Destination_Station',
     'Total_Distance_km', 'Journey_Duration_Hrs', 'Avg_Speed_kmh', 'Num_Stops']
].reset_index(drop=True)
fastest.index += 1
print(fastest.to_string())

⚡ Top 10 Fastest Trains (by Avg Speed):
    Train_No Source_Station Destination_Station  Total_Distance_km  Journey_Duration_Hrs  Avg_Speed_kmh  Num_Stops
1      11463        SOMNATH            JABALPUR               1416                  5.67         249.74         50
2       8064   YESVANTPUR J          HOWRAH JN.               2039                  8.17         249.57         16
3      12622      NEW DELHI        CHENNAI CENT               2163                  8.67         249.48         11
4      14317      INDORE BG           DEHRA DUN               1165                  4.67         249.46         30
5       7439   TATANAGAR JN           KACHEGUDA               1533                  6.17         248.46         28
6      12925   BANDRA TERMI        AMRITSAR JN.               1810                  7.33         246.93         42
7      11464       JABALPUR             SOMNATH               1417                  5.75         246.43         50
8      12904   AMRITSAR JN.        MUMBA

In [14]:
# Most stops trains
print('🛑 Top 10 Trains with Most Stops:')
print('=' * 70)
most_stops = train_master.nlargest(10, 'Num_Stops')[
    ['Train_No', 'Source_Station', 'Destination_Station',
     'Num_Stops', 'Total_Distance_km', 'Journey_Duration_Hrs']
].reset_index(drop=True)
most_stops.index += 1
print(most_stops.to_string())

🛑 Top 10 Trains with Most Stops:
    Train_No Source_Station Destination_Station  Num_Stops  Total_Distance_km  Journey_Duration_Hrs
1      53041     HOWRAH JN.            JAYNAGAR        118                718                  5.33
2      13007     HOWRAH JN.        SHRI GANGANA        112               1949                 21.42
3      13049     HOWRAH JN.        AMRITSAR JN.        111               1908                 20.50
4      13008   SHRI GANGANA          HOWRAH JN.        108               1952                 22.50
5      13050   AMRITSAR JN.          HOWRAH JN.        106               1909                 21.58
6      19019   BANDRA TERMI           DEHRA DUN         98               1676                 17.50
7      13152     JAMMU TAWI             KOLKATA         97               1979                 20.83
8      19024   FIROZPUR CAN        MUMBAI CENTR         97               1762                 14.58
9      13151        KOLKATA          JAMMU TAWI         96         

In [15]:
# Final insights summary
print('=' * 65)
print('        💡 LEVEL 3 — KEY EDA INSIGHTS')
print('=' * 65)
print(f"  Total Trains Analyzed        : {len(train_master):,}")
print(f"  Total Stations               : {len(station_traffic):,}")
print()
print('  🔵 Journey Duration:')
print(f"    → Avg Duration             : {train_master['Journey_Duration_Hrs'].mean():.2f} hrs")
print(f"    → Shortest Journey         : {train_master['Journey_Duration_Hrs'].min():.2f} hrs")
print(f"    → Longest Journey          : {train_master['Journey_Duration_Hrs'].max():.2f} hrs")
print(f"    → Most trains are Short/Medium (<8 hrs)")
print()
print('  🔵 Station Traffic:')
busiest = station_traffic.iloc[0]
print(f"    → Busiest Station          : {busiest['Station_Name']} ({busiest['Train_Count']} trains)")
print(f"    → Many stations serve only 1–2 trains (rural/remote)")
print()
print('  🔵 Distance vs Duration:')
print(f"    → Correlation              : {corr_dist_dur:.4f} (Strong Positive)")
print(f"    → Longer distance = longer journey (obvious but confirmed!)")
print(f"    → Avg Speed varies: slower for more stops, faster for express")
print()
print('  🔵 ML Model Readiness:')
print(f"    → Best features: Total_Distance_km + Num_Stops")
print(f"    → Target: Journey_Duration_Min")
print(f"    → Ready for Level 4 (Visualization) & Level 5 (ML Model)")
print('=' * 65)

        💡 LEVEL 3 — KEY EDA INSIGHTS
  Total Trains Analyzed        : 10,625
  Total Stations               : 8,150

  🔵 Journey Duration:
    → Avg Duration             : 4.64 hrs
    → Shortest Journey         : 0.08 hrs
    → Longest Journey          : 23.92 hrs
    → Most trains are Short/Medium (<8 hrs)

  🔵 Station Traffic:
    → Busiest Station          : CST-MUMBAI (1027 trains)
    → Many stations serve only 1–2 trains (rural/remote)

  🔵 Distance vs Duration:
    → Correlation              : 0.8025 (Strong Positive)
    → Longer distance = longer journey (obvious but confirmed!)
    → Avg Speed varies: slower for more stops, faster for express

  🔵 ML Model Readiness:
    → Best features: Total_Distance_km + Num_Stops
    → Target: Journey_Duration_Min
    → Ready for Level 4 (Visualization) & Level 5 (ML Model)
